## 0. Load training data

In [1]:
import sys
sys.path.insert(0, '..')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
from pathlib import Path


#from src.data import load_raw, basic_clean, split_xy, ID_COL, TARGET
#from src.features import add_all_features

def load_churn_train_data():
    # Load the data from the local file, or download it if it doesn't exist
    churn_path = Path("churn_cleaned_train_set.csv")
    #throw an error if the file doesn't exist
    if not churn_path.is_file():
        raise FileNotFoundError("File not found: churn_cleaned_train_set.csv")
    return pd.read_csv(churn_path)

churn_training = load_churn_train_data()

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 80)

In [2]:
#code to save the figures as high-res PNGs

IMAGES_PATH = Path() / "images" / "model_testing"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [3]:
#the next 5 lines define the default font sizes of graphs
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## 1. Transformation Pipeline

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

In [5]:
from sklearn.pipeline import make_pipeline

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

In [6]:
from sklearn import set_config

set_config(display='diagram')

num_pipeline

,steps,"[('simpleimputer', ...), ('standardscaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [7]:
churn_num_prepared = num_pipeline.fit_transform(churn_training)
churn_num_prepared[:2].round(2)

ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: '2024-06-18'

## 1. Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

## 1 .XGBoost

In [1]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import shap

# Class imbalance: scale_pos_weight ≈ neg / pos in your training data
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=neg / pos,
    eval_metric="auc",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

probs = xgb.predict_proba(X_test)[:, 1]
print(f"Test ROC-AUC:        {roc_auc_score(y_test, probs):.3f}")
print(f"Best iteration:      {xgb.best_iteration}")

# Explain individual predictions with SHAP — critical for stakeholder trust
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, plot_type="bar")

NameError: name 'y_train' is not defined